In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import anndata as ad

import NaiveDE
import SpatialDE

from datetime import date
today = date.today()

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

HERE = os.path.dirname(os.path.abspath(__file__))
DEFAULT_OUTPUT_ROOT = os.path.join(HERE, "results")

Helpers to load data and save results

In [1]:
def load_visium(path) -> ad.AnnData:
    """
    Load a single Visium sample into an AnnData with obsm["spatial"]
    """
    adata = ad.read_h5ad(os.path.join(path, 'adata.h5ad'))
    adata.var_names_make_unique()
    
    return adata

def _unique_dir(path: str) -> str:
    """`path`, or `path` with a numeric suffix (_1, _2, ...) appended to
    its final component if it already exists -- run directories are
    never overwritten."""
    if not os.path.exists(path):
        return path
    parent, name = os.path.split(path)
    i = 1
    while True:
        candidate = os.path.join(parent, f"{name}_{i}")
        if not os.path.exists(candidate):
            return candidate
        i += 1



NameError: name 'ad' is not defined

Load data and make output directory

In [ ]:
output_root = 'results'
dataset_id = 'libd'
sample_id = '151673'
run_id = 'spatialde'

Use same genes as in main text

In [ ]:
data_path = os.path.join("../data/", sample_id)

In [ ]:
base_dir = os.path.join(output_root, dataset_id, sample_id, run_id)
out_dir = _unique_dir(base_dir)
os.makedirs(out_dir)

print(f"Writing results to {out_dir} on {today}", flush=True)

print(f"Loading Visium sample from {args.path}", flush=True)
adata = load_visium(data_path)

Process data as in the notebooks [here](https://github.com/Teichlab/SpatialDE/tree/master/Analysis)

In [ ]:
sample_info = adata.obs
sample_info = sample_info.query('total_counts > 10')  # Remove empty features

X = sample_info[['array_row', 'array_col']].values
df = pd.DataFrame(adata.X.toarray(), columns=adata.var_names, index=adata.obs_names)

dfm = NaiveDE.stabilize(df.T).T
res = NaiveDE.regress_out(sample_info, dfm.T, 'np.log(total_counts)').T

Run spatialDE

In [ ]:
de_results = results[(results.qval < 0.05)].copy()
de_results.to_csv(os.path.join(out_dir, 'de_results.csv'))

print(f"Fit finished in {fit_seconds:.1f}s", flush=True)


t0 = time.perf_counter()
ms_results = SpatialDE.model_search(X, res, de_results)
fit_seconds = time.perf_counter() - t0

ms_results.to_csv(os.path.join(out_dir, 'model_search.csv'))

print(f"Model search finished in {fit_seconds:.1f}s", flush=True)

print(f"Done. Results written to {out_dir}", flush=True)